# Práctica 1: Listas Doblemente Enlazadas (`DList`)

## Objetivos

En esta práctica trabajarás con una **lista doblemente enlazada**, en la que cada nodo mantiene referencias al nodo anterior (`prev`) y al siguiente (`next`).

A lo largo de los retos practicarás:

- La eliminación segura de nodos manteniendo coherentes `prev`, `next`, `_head`, `_tail` y `_size`.
- El recorrido bidireccional desde ambos extremos de la lista.
- La inversión *in-place* de una `DList`.
- El intercambio físico de nodos adyacentes.
- El recorrido inverso desde `_tail` mediante referencias `prev`.
- La aplicación de una lista doblemente enlazada a un historial de navegación web.
- La comprobación automática de tus soluciones mediante `assert`.

## ¿Cómo realizar esta práctica?

La práctica está organizada en **cuatro retos**.

Cada reto incluye:

- 🎯 **Objetivo**
- 🕒 **Tiempo estimado**
- 🟢🟡🔴 **Dificultad**
- 📝 **Enunciado**
- 💡 **Pista**, cuando resulte útil
- 💻 Una celda para escribir la solución
- ✅ Una celda independiente con pruebas automáticas

Ejecuta las celdas **en orden**, comenzando por las clases base `DNode` y `DList`.

> ## 💡 Recomendación
>
> Antes de modificar una `DList`, dibuja los nodos y sus referencias `prev` y `next`. Comprueba siempre qué sucede con `_head`, `_tail` y `_size`.

> **📝 Nota**
>
> No modifiques las clases base `DNode` y `DList`.
>
> En todo momento debe mantenerse la coherencia bidireccional: si `a.next is b`, entonces `b.prev is a`.

## Comprobación automática

Si todos los `assert` de un reto se ejecutan correctamente aparecerá:

`Pruebas superadas`

Si alguno falla, revisa la implementación y vuelve a ejecutar primero la celda de solución y después la celda de pruebas.

# 🧱 Código base: `DNode` y `DList`

`DList` utiliza:

- `_head`: primer nodo.
- `_tail`: último nodo.
- `_size`: número de nodos.
- `next` y `prev`: referencias entre nodos consecutivos.

**Ejecuta esta celda antes de comenzar y no la modifiques.**

In [ ]:
class DNode:
    """Nodo para una lista doblemente enlazada."""
    def __init__(self, elem, next=None, prev=None):
        self.elem = elem
        self.next = next
        self.prev = prev


class DList:
    """Implementación de una lista doblemente enlazada con referencias head y tail."""
    def __init__(self):
        self._head = None
        self._tail = None
        self._size = 0

    def __len__(self):
        return self._size

    def isEmpty(self):
        return len(self) == 0

    def __str__(self):
        nodeIt = self._head
        result = '['
        while nodeIt:
            if type(nodeIt.elem) == int:
                result += str(nodeIt.elem) + ", "
            else:
                result += "'" + str(nodeIt.elem) + "', "
            nodeIt = nodeIt.next
        if len(result) > 1:
            result = result[:-2]
        result += ']'
        return result

    def addFirst(self, e):
        newNode = DNode(e)
        if self.isEmpty():
            self._tail = newNode
        else:
            newNode.next = self._head
            self._head.prev = newNode
        self._head = newNode
        self._size += 1

    def addLast(self, e):
        newNode = DNode(e)
        if self.isEmpty():
            self._head = newNode
        else:
            newNode.prev = self._tail
            self._tail.next = newNode
        self._tail = newNode
        self._size += 1

    def removeFirst(self):
        result = None
        if self.isEmpty():
            print("Error: list is empty")
        else:
            result = self._head.elem
            self._head = self._head.next
            if self._head is None:
                self._tail = None
            else:
                self._head.prev = None
            self._size -= 1
        return result

    def removeLast(self):
        result = None
        if self.isEmpty():
            print("Error: list is empty")
        else:
            result = self._tail.elem
            self._tail = self._tail.prev
            if self._tail is None:
                self._head = None
            else:
                self._tail.next = None
            self._size -= 1
        return result

    def getAt(self, index):
        if index < 0 or index >= self._size:
            print(index, 'Error: index out of range')
            return None
        if index <= len(self) // 2:
            i = 0
            current = self._head
            while i < index:
                current = current.next
                i += 1
        else:
            i = 0
            current = self._tail
            while i < (len(self) - 1) - index:
                current = current.prev
                i += 1
        return current.elem

    def index(self, e):
        nodeIt = self._head
        idx = 0
        while nodeIt:
            if nodeIt.elem == e:
                return idx
            nodeIt = nodeIt.next
            idx += 1
        return -1

    def insertAt(self, index, e):
        if index not in range(len(self) + 1):
            print('Error: index out of range')
        elif index == 0:
            self.addFirst(e)
        elif index == len(self):
            self.addLast(e)
        else:
            nodeIt = self._head
            for _ in range(index):
                nodeIt = nodeIt.next
            previous = nodeIt.prev
            newNode = DNode(e)
            newNode.next = nodeIt
            newNode.prev = previous
            previous.next = newNode
            nodeIt.prev = newNode
            self._size += 1

    def removeAt(self, index):
        result = None
        if index not in range(len(self)):
            print(index, 'Error removeAt: index out of range')
        elif index == 0:
            result = self.removeFirst()
        elif index == len(self) - 1:
            result = self.removeLast()
        else:
            nodeIt = self._head
            for _ in range(index):
                nodeIt = nodeIt.next
            result = nodeIt.elem
            prevNode = nodeIt.prev
            nextNode = nodeIt.next
            prevNode.next = nextNode
            nextNode.prev = prevNode
            self._size -= 1
        return result

# 🏆 Reto 1. Eliminar todos los números impares

🎯 **Objetivo:** Corregir un algoritmo que elimina nodos sin mantener correctamente las referencias dobles.

🕒 **Tiempo estimado:** 20 minutos.

🔴 **Dificultad:** Avanzada.

## 📝 Enunciado

Un estudiante intentó implementar `eliminar_impares_incorrecto(lista)` para eliminar todos los enteros impares de una `DList`.

El código presenta tres fallos principales:

1. No actualiza correctamente `prev`.
2. Puede provocar `AttributeError` al eliminar `_head` o `_tail`.
3. No mantiene correctamente `_head`, `_tail` y `_size`.

Implementa:

```python
eliminar_impares_corregido(lista: DList)
```

La función debe:

- Modificar la lista **in-place**.
- Eliminar todos los enteros impares.
- Mantener el orden relativo de los elementos pares.
- Mantener coherentes `_head`, `_tail`, `next`, `prev` y `_size`.

> **💡 Pista**
>
> Antes de modificar un nodo, guarda una referencia a su nodo siguiente. Después decide por separado cómo reconectar el predecesor y el sucesor.

### 🔎 Código incorrecto que debes analizar

In [ ]:
def eliminar_impares_incorrecto(lista: DList):
    actual = lista._head
    while actual:
        if actual.elem % 2 != 0:
            actual.prev.next = actual.next
            actual.next.prev = actual.prev
            lista._size -= 1
        actual = actual.next

### 💻 Tu solución

In [ ]:
def eliminar_impares_corregido(lista: DList):
    """
    Elimina todos los enteros impares manteniendo coherentes
    _head, _tail, next, prev y _size.
    """
    # TODO: implementa la versión corregida
    pass

### ✅ Comprobación automática

In [ ]:
dl1 = DList()
for x in [1, 3, 2, 5, 4, 7, 8, 9]:
    dl1.addLast(x)

eliminar_impares_corregido(dl1)

assert str(dl1) == "[2, 4, 8]"
assert len(dl1) == 3
assert dl1._head.elem == 2 and dl1._head.prev is None
assert dl1._tail.elem == 8 and dl1._tail.next is None

# Todos impares.
solo_impares = DList()
for x in [1, 3, 5]:
    solo_impares.addLast(x)

eliminar_impares_corregido(solo_impares)
assert solo_impares.isEmpty()
assert solo_impares._head is None and solo_impares._tail is None

print("Pruebas superadas")

# 🏆 Reto 2. Verificar si una `DList` es palíndroma

🎯 **Objetivo:** Aprovechar el recorrido bidireccional de una lista doblemente enlazada.

🕒 **Tiempo estimado:** 15 minutos.

🟡 **Dificultad:** Intermedia.

## 📝 Enunciado

Implementa:

```python
es_palindromo(lista: DList) -> bool
```

La función debe devolver `True` cuando la secuencia de elementos sea igual al recorrerla desde `_head` hacia `_tail` y desde `_tail` hacia `_head`.

Ejemplos:

- `[1, 2, 3, 2, 1]` → `True`
- `['a', 'b', 'b', 'a']` → `True`
- `[1, 2, 3, 4]` → `False`

Requisitos:

- Utiliza **dos referencias**, una desde `_head` y otra desde `_tail`.
- Ambas deben avanzar simultáneamente hacia el centro.
- No conviertas la estructura a una lista nativa de Python.
- Una lista vacía o con un único elemento devuelve `True`.

> **💡 Pista**
>
> El recorrido finaliza cuando ambos punteros coinciden o se cruzan.

### 💻 Tu solución

In [ ]:
def es_palindromo(lista: DList) -> bool:
    """
    Retorna True si los elementos forman una secuencia palíndroma.
    """
    # TODO: utiliza un recorrido bidireccional desde ambos extremos
    pass

### ✅ Comprobación automática

In [ ]:
p1 = DList()
for item in ['r', 'a', 'd', 'a', 'r']:
    p1.addLast(item)

p2 = DList()
for item in [1, 2, 3, 4]:
    p2.addLast(item)

p3 = DList()
for item in [1, 2, 2, 1]:
    p3.addLast(item)

assert es_palindromo(p1) is True
assert es_palindromo(p2) is False
assert es_palindromo(p3) is True
assert es_palindromo(DList()) is True

un_elemento = DList()
un_elemento.addLast("X")
assert es_palindromo(un_elemento) is True

print("Pruebas superadas")

# 🏆 Reto 3. Extensión avanzada de `DList`

🎯 **Objetivo:** Implementar operaciones que explotan directamente las referencias `prev` y `next`.

🕒 **Tiempo estimado:** 30 minutos.

🔴 **Dificultad:** Avanzada.

## 📝 Enunciado

Crea una subclase `DListAvanzada` que herede de `DList` e implemente tres métodos.

### `invertir()`

Invierte la lista **in-place**:

- Intercambia `next` y `prev` en cada nodo.
- Intercambia `_head` y `_tail`.
- No crea una nueva lista.

### `to_reverse_list() -> list`

- Recorre desde `_tail` hasta `_head` utilizando únicamente `prev`.
- Devuelve una lista nativa de Python con los elementos en orden inverso.
- No modifica la `DList`.

### `swap_adjacent(index) -> bool`

Intercambia **físicamente los nodos** en `index` e `index + 1`.

- Debe reconectar los punteros, no limitarse a intercambiar `elem`.
- Devuelve `False` si `index < 0` o `index >= len(self) - 1`.
- Debe actualizar `_head` o `_tail` cuando el intercambio afecta a un extremo.
- Devuelve `True` si se realiza el intercambio.

> **💡 Pista**
>
> En `swap_adjacent`, identifica cuatro referencias: el nodo anterior al par, los dos nodos a intercambiar y el nodo posterior al par.

### 💻 Tu solución

In [ ]:
class DListAvanzada(DList):
    """Subclase de DList con manipulación avanzada de punteros dobles."""

    def invertir(self):
        # TODO
        pass

    def to_reverse_list(self) -> list:
        # TODO
        pass

    def swap_adjacent(self, index: int) -> bool:
        # TODO
        pass

### ✅ Comprobación automática

In [ ]:
dla = DListAvanzada()
for val in [10, 20, 30, 40]:
    dla.addLast(val)

assert dla.to_reverse_list() == [40, 30, 20, 10]

assert dla.swap_adjacent(1) is True
assert str(dla) == "[10, 30, 20, 40]"
assert dla.swap_adjacent(-1) is False
assert dla.swap_adjacent(3) is False

dla.invertir()
assert str(dla) == "[40, 20, 30, 10]"
assert dla._head.elem == 40 and dla._tail.elem == 10
assert dla._head.prev is None and dla._tail.next is None

print("Pruebas superadas")

# 🏆 Reto 4. Historial de navegación web

🎯 **Objetivo:** Aplicar una `DList` a un historial que permita moverse hacia atrás y hacia adelante.

🕒 **Tiempo estimado:** 25 minutos.

🔴 **Dificultad:** Avanzada.

## 📝 Enunciado

Diseña la clase `NavegadorWeb`.

La clase mantiene:

```python
self.historial = DList()
self.actual
```

donde `actual` referencia el nodo de la página activa.

Implementa:

### `visitar_pagina(url)`

- Añade una nueva página inmediatamente después de la página actual.
- Si existen páginas en el historial futuro, deben descartarse.
- La nueva página pasa a ser la página actual.
- `_tail` y `_size` deben reflejar el historial realmente conservado.

### `atras()`

- Si existe `actual.prev`, mueve `actual` al nodo anterior y devuelve su URL.
- Si no existe página anterior, no cambia el estado y devuelve `None`.

### `adelante()`

- Si existe `actual.next`, mueve `actual` al nodo siguiente y devuelve su URL.
- Si no existe página posterior, no cambia el estado y devuelve `None`.

### `obtener_actual()`

Devuelve la URL de la página activa.

> **💡 Pista**
>
> Cuando visitas una página después de haber retrocedido, basta con enlazar `actual.next` con el nuevo nodo y convertirlo en la nueva `_tail`; el antiguo historial futuro deja de formar parte de la lista.

### 💻 Tu solución

In [ ]:
class NavegadorWeb:
    def __init__(self, pagina_inicio: str):
        self.historial = DList()
        self.historial.addLast(pagina_inicio)
        self.actual = self.historial._head

    def visitar_pagina(self, url: str):
        # TODO
        pass

    def atras(self):
        # TODO
        pass

    def adelante(self):
        # TODO
        pass

    def obtener_actual(self) -> str:
        if self.actual:
            return self.actual.elem
        return None

### ✅ Comprobación automática

In [ ]:
nav = NavegadorWeb("home.com")
nav.visitar_pagina("google.com")
nav.visitar_pagina("github.com")

assert nav.atras() == "google.com"
assert nav.atras() == "home.com"
assert nav.atras() is None

assert nav.adelante() == "google.com"

# Visitar aquí debe descartar "github.com".
nav.visitar_pagina("stackoverflow.com")

assert nav.obtener_actual() == "stackoverflow.com"
assert nav.adelante() is None
assert nav.historial._tail.elem == "stackoverflow.com"
assert str(nav.historial) == "['home.com', 'google.com', 'stackoverflow.com']"
assert len(nav.historial) == 3

assert nav.atras() == "google.com"

print("Pruebas superadas")

# 🎉 ¡Enhorabuena!

Has completado la **Práctica 1 de Listas Doblemente Enlazadas**.

## ¿Qué deberías haber aprendido?

- ✅ Eliminar nodos manteniendo enlaces dobles coherentes.
- ✅ Recorrer una lista desde ambos extremos.
- ✅ Invertir una `DList` *in-place*.
- ✅ Intercambiar físicamente nodos adyacentes.
- ✅ Recorrer hacia atrás mediante `prev`.
- ✅ Modelar un historial de navegación con una estructura bidireccional.
- ✅ Validar automáticamente tus implementaciones.